# 1-3. 얼굴 crop 품질 필터 수집기

MediaPipe Face Landmark를 보조 신호로 사용해 `Attentive`, `Drowsy`, `LookingAway` 얼굴 crop 데이터셋을 수집합니다. 저장 이미지는 ResNet/MobileNet 전이학습에 사용할 수 있도록 얼굴 주변 여백을 포함한 224x224 BGR crop입니다.


## 1. 라이브러리 불러오기


In [1]:
import csv
import time
from datetime import datetime
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_styles, drawing_utils

## 2. 설정


In [2]:
WINDOW_NAME = "Focus On Class - Face Crop Collector"
CAMERA_INDEX = 0
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

SAVE_ROOT = Path("dataset")
CLASS_LABELS = ["Attentive", "Drowsy", "LookingAway"]
SAVE_DIRS = {label: SAVE_ROOT / label for label in CLASS_LABELS}
METADATA_PATH = SAVE_ROOT / "metadata.csv"

for save_dir in SAVE_DIRS.values():
    save_dir.mkdir(parents=True, exist_ok=True)

CROP_OUTPUT_SIZE = (224, 224)
FACE_PADDING_RATIO = 0.25
MIN_FACE_AREA_RATIO = 0.03
MIN_BLUR_THRESHOLD = 60.0
MIN_BRIGHTNESS = 40.0

BLUE_OVERLAY_ALPHA = 0.25
FACE_BOX_COLOR = (0, 255, 255)
TEXT_COLOR = (255, 255, 255)
WARNING_COLOR = (255, 220, 80)

FACE_MODEL_PATH = r"mp_model/face_landmarker.task"
HAND_MODEL_PATH = r"mp_model/hand_landmarker.task"
POSE_MODEL_PATH = r"mp_model/pose_landmarker_full.task"

BaseOptions = mp.tasks.BaseOptions
RunningMode = mp.tasks.vision.RunningMode
FaceLandmarker = mp.tasks.vision.FaceLandmarker
HandLandmarker = mp.tasks.vision.HandLandmarker
PoseLandmarker = mp.tasks.vision.PoseLandmarker

face_options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=FACE_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

hand_options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

pose_options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=True,
)


## 3. 기본 유틸 함수


In [3]:
def make_mp_image(frame_rgb: np.ndarray) -> mp.Image:
    """RGB 프레임을 MediaPipe 이미지 형식으로 변환합니다."""
    return mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(frame_rgb),
    )


def clamp01(x: float) -> float:
    return float(np.clip(x, 0.0, 1.0))


def _landmark_xy(face_landmarks, index: int, width: int, height: int) -> np.ndarray:
    lm = face_landmarks[index]
    return np.array([lm.x * width, lm.y * height], dtype=np.float32)


def _point_distance(p1: np.ndarray, p2: np.ndarray) -> float:
    return float(np.linalg.norm(p1 - p2))


def face_landmarks_to_bbox(face_landmarks, frame_width: int, frame_height: int):
    """측정과 화면 표시용으로 얼굴 랜드마크의 타이트한 박스를 만듭니다."""
    xs = [int(lm.x * frame_width) for lm in face_landmarks]
    ys = [int(lm.y * frame_height) for lm in face_landmarks]

    x1 = max(0, min(xs))
    y1 = max(0, min(ys))
    x2 = min(frame_width - 1, max(xs))
    y2 = min(frame_height - 1, max(ys))

    return x1, y1, x2, y2


def eye_open_score(face_landmarks, width, height):
    """양쪽 눈꺼풀 간격을 눈 너비로 정규화해 눈 뜨임 정도를 추정합니다."""
    left_top = _landmark_xy(face_landmarks, 159, width, height)
    left_bottom = _landmark_xy(face_landmarks, 145, width, height)
    left_outer = _landmark_xy(face_landmarks, 33, width, height)
    left_inner = _landmark_xy(face_landmarks, 133, width, height)

    right_top = _landmark_xy(face_landmarks, 386, width, height)
    right_bottom = _landmark_xy(face_landmarks, 374, width, height)
    right_inner = _landmark_xy(face_landmarks, 362, width, height)
    right_outer = _landmark_xy(face_landmarks, 263, width, height)

    left_ratio = _point_distance(left_top, left_bottom) / max(1.0, _point_distance(left_outer, left_inner))
    right_ratio = _point_distance(right_top, right_bottom) / max(1.0, _point_distance(right_inner, right_outer))
    eye_ratio = (left_ratio + right_ratio) / 2.0

    return clamp01((eye_ratio - 0.12) / 0.18)


def face_yaw_score(face_landmarks, width):
    """눈 중심 대비 코 위치 변화로 좌우 얼굴 회전 정도를 추정합니다."""
    left_eye_x = face_landmarks[33].x * width
    right_eye_x = face_landmarks[263].x * width
    nose_x = face_landmarks[1].x * width
    eye_center_x = (left_eye_x + right_eye_x) / 2.0
    eye_width = max(1.0, abs(right_eye_x - left_eye_x))

    return clamp01(abs(nose_x - eye_center_x) / (eye_width * 0.55))


def head_tilt_score(face_landmarks, width, height):
    """양쪽 눈꼬리를 잇는 선의 각도로 고개 기울어짐을 추정합니다."""
    left_eye = _landmark_xy(face_landmarks, 33, width, height)
    right_eye = _landmark_xy(face_landmarks, 263, width, height)
    dx, dy = right_eye - left_eye
    angle = abs(np.degrees(np.arctan2(dy, dx)))

    return clamp01(angle / 25.0)


def pose_head_down_score(pose_result, face_box, width, height):
    """얼굴 중심이 양쪽 어깨에 가까워진 정도로 포즈 기반 고개 숙임을 추정합니다."""
    pose_landmarks_list = getattr(pose_result, "pose_landmarks", []) if pose_result is not None else []
    if not pose_landmarks_list or face_box is None:
        return 0.0

    pose_landmarks = pose_landmarks_list[0]
    if len(pose_landmarks) <= 12:
        return 0.0

    left_shoulder = pose_landmarks[11]
    right_shoulder = pose_landmarks[12]
    shoulder_width = abs((right_shoulder.x - left_shoulder.x) * width)
    if shoulder_width < 1.0:
        return 0.0

    x1, y1, x2, y2 = face_box
    face_center_y = (y1 + y2) / 2.0
    shoulder_center_y = ((left_shoulder.y + right_shoulder.y) / 2.0) * height
    head_to_shoulder_ratio = (shoulder_center_y - face_center_y) / shoulder_width

    return clamp01((0.55 - head_to_shoulder_ratio) / 0.35)


def pose_only_drowsy_score(pose_result, width, height):
    """얼굴 검출이 실패했을 때 Pose의 얼굴/어깨 점만으로 크게 숙인 자세를 추정합니다."""
    pose_landmarks_list = getattr(pose_result, "pose_landmarks", []) if pose_result is not None else []
    if not pose_landmarks_list:
        return 0.0

    pose_landmarks = pose_landmarks_list[0]
    if len(pose_landmarks) <= 12:
        return 0.0

    def visible_point(index, min_visibility=0.35):
        landmark = pose_landmarks[index]
        visibility = getattr(landmark, "visibility", 1.0)
        if visibility < min_visibility:
            return None
        return np.array([landmark.x * width, landmark.y * height], dtype=np.float32)

    left_shoulder = visible_point(11, min_visibility=0.25)
    right_shoulder = visible_point(12, min_visibility=0.25)
    if left_shoulder is None or right_shoulder is None:
        return 0.0

    shoulder_width = _point_distance(left_shoulder, right_shoulder)
    if shoulder_width < 1.0:
        return 0.0

    face_points = [visible_point(index) for index in (0, 1, 2, 3, 4, 5, 6, 9, 10)]
    face_points = [point for point in face_points if point is not None]
    if not face_points:
        return 0.0

    face_center = np.mean(face_points, axis=0)
    shoulder_center = (left_shoulder + right_shoulder) / 2.0
    head_to_shoulder_ratio = (shoulder_center[1] - face_center[1]) / shoulder_width
    side_collapse = abs(face_center[0] - shoulder_center[0]) / max(1.0, shoulder_width)

    vertical_drop = clamp01((0.60 - head_to_shoulder_ratio) / 0.35)
    side_drop = clamp01((side_collapse - 0.25) / 0.45)
    return clamp01(max(vertical_drop, 0.65 * vertical_drop + 0.35 * side_drop))


def crop_face_with_padding(frame, face_box, padding_ratio=FACE_PADDING_RATIO):
    if face_box is None:
        return None

    height, width = frame.shape[:2]
    x1, y1, x2, y2 = face_box
    box_w = max(1, x2 - x1)
    box_h = max(1, y2 - y1)
    pad_x = int(box_w * padding_ratio)
    pad_y = int(box_h * padding_ratio)

    crop_x1 = max(0, x1 - pad_x)
    crop_y1 = max(0, y1 - pad_y)
    crop_x2 = min(width - 1, x2 + pad_x)
    crop_y2 = min(height - 1, y2 + pad_y)

    crop = frame[crop_y1:crop_y2 + 1, crop_x1:crop_x2 + 1]
    if crop.size == 0:
        return None

    return cv2.resize(crop, CROP_OUTPUT_SIZE, interpolation=cv2.INTER_AREA)


def is_valid_crop(crop):
    if crop is None or crop.size == 0:
        return False, "empty crop", 0.0, 0.0

    gray_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    blur_score = cv2.Laplacian(gray_crop, cv2.CV_64F).var()
    brightness = gray_crop.mean()

    if blur_score < MIN_BLUR_THRESHOLD:
        return False, f"blur {blur_score:.1f} < {MIN_BLUR_THRESHOLD:.1f}", blur_score, brightness
    if brightness < MIN_BRIGHTNESS:
        return False, f"brightness {brightness:.1f} < {MIN_BRIGHTNESS:.1f}", blur_score, brightness

    return True, "ok", blur_score, brightness


## 4. 라벨 추천 함수

Face Landmark feature를 0~1 범위로 정규화한 뒤, 복잡한 가중치 점수가 아니라 명확한 조건식으로 추천 라벨을 계산합니다.


In [4]:
def calculate_class_scores(face_result, face_boxes, frame_shape, pose_result=None):
    """규칙 기반 클래스 후보와 정규화된 얼굴 feature를 계산합니다."""
    scores = {"Attentive": 0.0, "Drowsy": 0.0, "LookingAway": 0.0}
    features = {
        "face_area": 0.0,
        "eye_open": 0.0,
        "head_down": 0.0,
        "pose_head_down": 0.0,
        "off_center": 1.0,
        "face_yaw": 1.0,
        "head_tilt": 1.0,
        "frontal": 0.0,
    }

    height, width = frame_shape[:2]
    face_landmarks_list = getattr(face_result, "face_landmarks", [])
    if not face_landmarks_list or not face_boxes:
        pose_head_down = pose_only_drowsy_score(pose_result, width, height)
        features["pose_head_down"] = pose_head_down
        scores["Drowsy"] = float(pose_head_down >= 0.60)
        return scores, features

    x1, y1, x2, y2 = face_boxes[0]
    box_w = max(1, x2 - x1)
    box_h = max(1, y2 - y1)

    face_area = (box_w * box_h) / max(1, width * height)
    face_size_score = clamp01(face_area / MIN_FACE_AREA_RATIO)

    face_center_x = (x1 + x2) / 2.0
    face_center_y = (y1 + y2) / 2.0
    center_dx = abs(face_center_x - width / 2.0) / (width / 2.0)
    center_dy = abs(face_center_y - height / 2.0) / (height / 2.0)
    off_center = clamp01((center_dx + center_dy) / 1.2)

    face_landmarks = face_landmarks_list[0]
    eye_open = eye_open_score(face_landmarks, width, height)

    nose_y = face_landmarks[1].y * height
    nose_ratio = (nose_y - y1) / max(1, box_h)
    # 정면 집중 상태에서도 코 위치가 얼굴 박스의 아래쪽으로 잡히는 경우가 있어
    # head_down은 보수적으로 계산합니다. 졸림 판정은 눈 감김을 핵심 기준으로 둡니다.
    head_down = clamp01((nose_ratio - 0.64) / 0.16)
    pose_head_down = pose_head_down_score(pose_result, face_boxes[0], width, height)

    face_yaw = face_yaw_score(face_landmarks, width)
    head_tilt = head_tilt_score(face_landmarks, width, height)
    frontal = clamp01(1.0 - max(face_yaw, head_tilt, off_center))

    is_attentive = (
        face_size_score >= 1.0 and
        eye_open >= 0.65 and
        head_down <= 0.25 and
        face_yaw <= 0.25 and
        head_tilt <= 0.25 and
        off_center <= 0.35
    )

    # 눈 landmark가 실패하는 크게 숙인 자세는 Pose를 보조 신호로 사용합니다.
    pose_drowsy_fallback = (
        head_down >= 0.55 and
        (
            pose_head_down >= 0.35 or
            head_tilt >= 0.30
        ) and
        face_yaw <= 0.45 and
        off_center <= 0.55
    )

    is_drowsy = (
        eye_open <= 0.25 or
        (eye_open <= 0.40 and head_down >= 0.45) or
        pose_drowsy_fallback
    )

    # 눈은 뜨고 있지만 고개가 기울어진 자세는 집중 저하로 보고 LookingAway로 추천합니다.
    posture_looking_away = (
        head_tilt >= 0.30 and
        pose_head_down >= 0.10
    )

    is_looking_away = (
        eye_open >= 0.50 and
        head_down <= 0.45 and
        (
            face_yaw >= 0.45 or
            off_center >= 0.55 or
            head_tilt >= 0.45 or
            posture_looking_away
        )
    )

    scores.update({
        "Attentive": float(is_attentive),
        "Drowsy": float(is_drowsy),
        "LookingAway": float(is_looking_away),
    })

    features.update({
        "face_area": clamp01(face_area),
        "eye_open": eye_open,
        "head_down": head_down,
        "pose_head_down": pose_head_down,
        "off_center": off_center,
        "face_yaw": face_yaw,
        "head_tilt": head_tilt,
        "frontal": frontal,
    })
    return scores, features


def get_recommended_label(scores):
    """여러 조건이 동시에 만족될 때 Drowsy > LookingAway > Attentive 우선순위로 추천 라벨을 정합니다."""
    for label in ("Drowsy", "LookingAway", "Attentive"):
        if scores.get(label, 0.0) >= 1.0:
            return label
    return None


## 5. 화면 표시 함수


In [5]:
def draw_blue_segmentation_overlay(frame_rgb: np.ndarray, pose_result) -> np.ndarray:
    annotated_frame = frame_rgb.copy()

    if not getattr(pose_result, "segmentation_masks", None):
        return annotated_frame

    segmentation_mask = np.squeeze(pose_result.segmentation_masks[0].numpy_view())
    if segmentation_mask.shape[:2] != annotated_frame.shape[:2]:
        segmentation_mask = cv2.resize(
            segmentation_mask,
            (annotated_frame.shape[1], annotated_frame.shape[0]),
            interpolation=cv2.INTER_LINEAR,
        )

    blue_overlay = np.zeros_like(annotated_frame, dtype=np.uint8)
    blue_overlay[:, :, 2] = (segmentation_mask * 255).astype(np.uint8)
    return cv2.addWeighted(annotated_frame, 1.0, blue_overlay, BLUE_OVERLAY_ALPHA, 0)


def draw_face_landmarks(frame_rgb: np.ndarray, face_result):
    annotated_frame = frame_rgb.copy()
    face_boxes = []
    height, width = annotated_frame.shape[:2]

    for face_idx, face_landmarks in enumerate(getattr(face_result, "face_landmarks", [])):
        x1, y1, x2, y2 = face_landmarks_to_bbox(face_landmarks, width, height)
        face_boxes.append((x1, y1, x2, y2))

        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), FACE_BOX_COLOR, 2)
        cv2.putText(annotated_frame, f"Face {face_idx + 1}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, TEXT_COLOR, 2, cv2.LINE_AA)
        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=face_landmarks,
            connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
            landmark_drawing_spec=None,
            connection_drawing_spec=drawing_styles.get_default_face_mesh_contours_style(),
        )

    return annotated_frame, face_boxes


def draw_hand_landmarks(frame_rgb: np.ndarray, hand_result) -> np.ndarray:
    annotated_frame = frame_rgb.copy()
    for hand_landmarks in getattr(hand_result, "hand_landmarks", []):
        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=hand_landmarks,
            connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
            landmark_drawing_spec=drawing_styles.get_default_hand_landmarks_style(),
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style(),
        )
    return annotated_frame


def draw_pose_landmarks(frame_rgb: np.ndarray, pose_result) -> np.ndarray:
    annotated_frame = frame_rgb.copy()
    for pose_landmarks in getattr(pose_result, "pose_landmarks", []):
        drawing_utils.draw_landmarks(
            image=annotated_frame,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=drawing_styles.get_default_pose_landmarks_style(),
            connection_drawing_spec=drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2),
        )
    return annotated_frame


def draw_status(frame_rgb: np.ndarray, fps: float, face_count: int, hand_count: int, recommended_label, saved_status, features) -> np.ndarray:
    annotated_frame = frame_rgb.copy()
    recommended_text = recommended_label if recommended_label is not None else "None"
    lines = [
        f"FPS: {fps:.1f}",
        f"Faces: {face_count}  Hands: {hand_count}",
        f"Recommended: {recommended_text}",
        f"Saved: {saved_status}",
        "1: Attentive | 2: Drowsy | 3: LookingAway | a: Auto | q: Quit",
        f"eye_open: {features['eye_open']:.2f}",
        f"head_down: {features['head_down']:.2f}",
        f"pose_head_down: {features.get('pose_head_down', 0.0):.2f}",
        f"face_yaw: {features['face_yaw']:.2f}",
        f"head_tilt: {features['head_tilt']:.2f}",
        f"off_center: {features['off_center']:.2f}",
        f"face_area: {features['face_area']:.3f}",
    ]

    y = 30
    for line in lines:
        color = WARNING_COLOR if line.startswith("Saved:") and "skipped" in line.lower() else TEXT_COLOR
        cv2.putText(annotated_frame, line, (20, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
        y += 24

    return annotated_frame


## 6. 저장 함수

저장 직전에 crop 품질을 검사하고, 통과한 224x224 얼굴 crop만 클래스 폴더에 기록합니다.


In [6]:
def append_metadata(rows):
    """저장한 이미지의 메타데이터를 CSV에 추가합니다."""
    if not rows:
        return

    METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    file_exists = METADATA_PATH.exists()
    fieldnames = [
        "path", "label", "recommended_label", "frame_count", "blur_score", "brightness",
        "face_area", "eye_open", "head_down", "pose_head_down", "off_center", "face_yaw", "head_tilt", "frontal", "timestamp",
    ]

    with METADATA_PATH.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)


def save_crop(crop, label, frame_count, features=None, recommended_label=None):
    """품질 검사를 통과한 224x224 crop 한 장을 저장하고 메타데이터를 기록합니다."""
    if label not in SAVE_DIRS:
        return False, f"unknown label: {label}"

    is_valid, reason, blur_score, brightness = is_valid_crop(crop)
    if not is_valid:
        return False, f"skipped: {reason}"

    save_dir = SAVE_DIRS[label]
    save_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    image_path = save_dir / f"{label}_{timestamp}_{frame_count:06d}.jpg"
    saved = cv2.imwrite(str(image_path), crop)
    if not saved:
        return False, f"save failed: {image_path}"

    row = {
        "path": str(image_path),
        "label": label,
        "recommended_label": recommended_label or "None",
        "frame_count": frame_count,
        "blur_score": round(blur_score, 4),
        "brightness": round(brightness, 4),
        "timestamp": timestamp,
    }
    if features:
        row.update({key: round(value, 4) for key, value in features.items()})

    append_metadata([row])
    return True, f"saved: {image_path}"


## 7. 저장 폴더 준비

메인 루프를 실행하기 전에 `dataset`과 라벨별 폴더를 다시 만듭니다. 수집 중에 폴더를 지웠더라도 저장 실패를 줄일 수 있습니다.


In [7]:
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

for label, save_dir in SAVE_DIRS.items():
    save_dir.mkdir(parents=True, exist_ok=True)
    print(f"ready: {label} -> {save_dir}")

print(f"metadata path: {METADATA_PATH}")

ready: Attentive -> dataset\Attentive
ready: Drowsy -> dataset\Drowsy
ready: LookingAway -> dataset\LookingAway
metadata path: dataset\metadata.csv


## 8. 메인 실행 루프

웹캠 창에서 `1`, `2`, `3`은 수동 라벨로 즉시 저장하고, `a`는 추천 라벨이 있을 때만 저장합니다. `q`를 누르면 안전하게 종료합니다.


In [17]:
cv2.destroyAllWindows()

with (
    PoseLandmarker.create_from_options(pose_options) as pose_landmarker,
    FaceLandmarker.create_from_options(face_options) as face_landmarker,
    HandLandmarker.create_from_options(hand_options) as hand_landmarker,
):
    camera_capture = cv2.VideoCapture(CAMERA_INDEX)
    camera_capture.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
    camera_capture.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

    if not camera_capture.isOpened():
        raise RuntimeError("Camera open failed. Check CAMERA_INDEX or camera permission.")

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    start_time = time.perf_counter()
    last_frame_time = time.perf_counter()
    frame_count = 0
    last_save_status = "ready"
    key_to_label = {
        ord("1"): "Attentive",
        ord("2"): "Drowsy",
        ord("3"): "LookingAway",
    }

    try:
        while True:
            is_readable, frame_bgr = camera_capture.read()
            if not is_readable:
                print("Frame read failed.")
                break

            frame_count += 1

            frame_bgr = cv2.flip(frame_bgr, 1)
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            mp_image = make_mp_image(frame_rgb)
            timestamp_ms = int((time.perf_counter() - start_time) * 1000)

            pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
            face_result = face_landmarker.detect_for_video(mp_image, timestamp_ms)
            hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

            annotated_frame_rgb = draw_blue_segmentation_overlay(frame_rgb, pose_result)
            annotated_frame_rgb = draw_pose_landmarks(annotated_frame_rgb, pose_result)
            annotated_frame_rgb, face_boxes = draw_face_landmarks(annotated_frame_rgb, face_result)
            annotated_frame_rgb = draw_hand_landmarks(annotated_frame_rgb, hand_result)

            scores, features = calculate_class_scores(face_result, face_boxes, frame_rgb.shape, pose_result=pose_result)
            recommended_label = get_recommended_label(scores)

            current_time = time.perf_counter()
            fps = 1.0 / max(1e-6, current_time - last_frame_time)
            last_frame_time = current_time

            hand_count = len(getattr(hand_result, "hand_landmarks", []))
            annotated_frame_rgb = draw_status(
                annotated_frame_rgb,
                fps,
                len(face_boxes),
                hand_count,
                recommended_label,
                last_save_status,
                features,
            )

            display_frame_bgr = cv2.cvtColor(annotated_frame_rgb, cv2.COLOR_RGB2BGR)
            cv2.imshow(WINDOW_NAME, display_frame_bgr)

            key = cv2.waitKey(1) & 0xFF

            if key in key_to_label or key == ord("a"):
                if not face_boxes:
                    last_save_status = "skipped: no face"
                    print(last_save_status)
                    continue

                if features["face_area"] < MIN_FACE_AREA_RATIO:
                    last_save_status = f"skipped: face_area {features['face_area']:.3f} < {MIN_FACE_AREA_RATIO:.3f}"
                    print(last_save_status)
                    continue

                if key == ord("a"):
                    if recommended_label is None:
                        last_save_status = "skipped: recommendation None"
                        print(last_save_status)
                        continue
                    label = recommended_label
                else:
                    label = key_to_label[key]

                crop = crop_face_with_padding(frame_bgr, face_boxes[0], padding_ratio=FACE_PADDING_RATIO)
                saved, message = save_crop(crop, label, frame_count, features=features.copy(), recommended_label=recommended_label)
                last_save_status = message
                print(message)
                continue

            if key == ord("q"):
                print("[q] quit.")
                break

    finally:
        camera_capture.release()
        cv2.destroyAllWindows()


saved: dataset\Drowsy\Drowsy_20260424_170913_000031.jpg
saved: dataset\Drowsy\Drowsy_20260424_170914_000039.jpg
saved: dataset\Drowsy\Drowsy_20260424_170914_000047.jpg
saved: dataset\Drowsy\Drowsy_20260424_170915_000066.jpg
saved: dataset\Drowsy\Drowsy_20260424_170916_000076.jpg
saved: dataset\Drowsy\Drowsy_20260424_170918_000119.jpg
saved: dataset\Drowsy\Drowsy_20260424_170919_000128.jpg
skipped: no face
saved: dataset\Drowsy\Drowsy_20260424_170922_000187.jpg
saved: dataset\Drowsy\Drowsy_20260424_170923_000197.jpg
skipped: no face
saved: dataset\Drowsy\Drowsy_20260424_170929_000317.jpg
saved: dataset\Drowsy\Drowsy_20260424_170930_000330.jpg
saved: dataset\Drowsy\Drowsy_20260424_170932_000374.jpg
saved: dataset\Drowsy\Drowsy_20260424_170933_000391.jpg
saved: dataset\Drowsy\Drowsy_20260424_170934_000403.jpg
[q] quit.


## 9. 수집 개수 확인


In [18]:
for label, save_dir in SAVE_DIRS.items():
    file_count = sum(1 for path in save_dir.glob("*.jpg"))
    print(f"{label}: {file_count}")

print(f"metadata: {METADATA_PATH}")


Attentive: 63
Drowsy: 64
LookingAway: 61
metadata: dataset\metadata.csv
